# Mengukur Jarak Tipe Data Campuran (Gower Distance)

Dalam analisis data mining, seringkali kita menemukan dataset yang berisi campuran antara fitur numerik (angka) dan kategorikal (teks/label). Untuk mengukur kemiripan antar objek pada tipe data ini, kita menggunakan **Gower Distance**.

Pada bagian ini, kita menggunakan dataset **Travel Insurance Prediction** yang memiliki atribut campuran seperti Umur, Pendapatan, dan Status Pekerjaan.

In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv('TravelInsurancePrediction.csv')

df_subset = df[['Age', 'Employment Type', 'GraduateOrNot', 'AnnualIncome', 'EverTravelledAbroad']].head(5).copy()

print("Sampel Data Pelanggan:")
display(df_subset)

Sampel Data Pelanggan:


,Age,Employment Type,GraduateOrNot,AnnualIncome,EverTravelledAbroad
0,31,Government Sector,Yes,400000,No
1,31,Private Sector/Self Employed,Yes,1250000,No
2,34,Private Sector/Self Employed,Yes,500000,No
3,28,Private Sector/Self Employed,Yes,700000,No
4,28,Private Sector/Self Employed,Yes,700000,No


## 1. Transformasi dan Rumus Manual

Untuk menghitung jarak Gower antara individu $i$ dan $j$, kita menghitung rata-rata tertimbang dari perbedaan di setiap fitur:

1.  **Fitur Numerik (Age, AnnualIncome):**
    Menggunakan selisih absolut dibagi dengan rentang (range) nilai fitur tersebut.
    $$s_{ij}^{(k)} = \frac{|x_{ik} - x_{jk}|}{R_k}$$
2.  **Fitur Kategorikal (Employment Type, Graduate, dll):**
    Bernilai 0 jika kategorinya sama, dan 1 jika berbeda.

### **Contoh Hitung Manual (Pelanggan 1 vs Pelanggan 2):**
* **Age:** $|31 - 31| / 10 = 0$
* **Employment Type:** Gov vs Private (Beda) = $1$
* **AnnualIncome:** $|400.000 - 1.250.000| / 1.500.000 = 0.566$
* **Total Jarak:** $\frac{0 + 1 + 0 + 0.566 + 0}{5} = \mathbf{0.3132}$

In [16]:
# Fungsi Perhitungan Jarak Gower secara otomatis
def calculate_gower(row1, row2, types, ranges):
    dist = 0
    for i, col in enumerate(row1.index):
        if types[i] == 'num':
            dist += abs(row1[col] - row2[col]) / ranges[col]
        else:
            dist += 1 if row1[col] != row2[col] else 0
    return dist / len(row1)

# Parameter (Range: Age=10, Income=1.5M)
ranges = {'Age': 10, 'AnnualIncome': 1500000}
types = ['num', 'cat', 'cat', 'num', 'cat']

# Membuat Matriks Jarak
n = len(df_subset)
matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        matrix[i,j] = calculate_gower(df_subset.iloc[i], df_subset.iloc[j], types, ranges)

# Tampilkan sebagai DataFrame
names = [f'Pelanggan {i+1}' for i in range(n)]
df_gower = pd.DataFrame(matrix, columns=names, index=names)

print("Matriks Jarak Gower (Hasil Python):")
display(df_gower)

Matriks Jarak Gower (Hasil Python):


,Pelanggan 1,Pelanggan 2,Pelanggan 3,Pelanggan 4,Pelanggan 5
Pelanggan 1,0.000000,0.313333,0.273333,0.300000,0.300000
Pelanggan 2,0.313333,0.000000,0.160000,0.133333,0.133333
Pelanggan 3,0.273333,0.160000,0.000000,0.146667,0.146667
Pelanggan 4,0.300000,0.133333,0.146667,0.000000,0.000000
Pelanggan 5,0.300000,0.133333,0.146667,0.000000,0.000000


## 2. Implementasi di Orange Data Mining

Karena keterbatasan beberapa versi tools dalam mendeteksi metrik Gower secara otomatis, perhitungan utama dilakukan menggunakan Python. Berikut adalah alur kerja (workflow) di Orange untuk memvalidasi pemrosesan data campuran:

![Workflow Orange](workflow_campuran.png)
 

![Gower Matrix](gower_matrix.png)
 